# Day 3: Batch Ingestion (X12 EDI Claims & Debezium CDC)

This harness parses synthetic raw EDI lines and Debezium JSON records into structured Bronze output.

In [ ]:
import os
from pyspark.sql import SparkSession

os.environ['SPARK_VERSION'] = '3.3'

spark = SparkSession.builder \
    .appName("Day03_Batch_CDC_Ingestion") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.3_2.12:1.3.1,org.apache.iceberg:iceberg-aws-bundle:1.3.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/ehdip_warehouse") \
    .getOrCreate()

print("SparkSession initialized for Batch Simulator.")

In [ ]:
# Create sample X12 and CDC files
import json

os.makedirs("/tmp/mock_data", exist_ok=True)

# 1. Mock X12 EDI
with open("/tmp/mock_data/sample.edi", "w") as f:
    f.write("ISA*00*          *00*          *ZZ*SENDERID       *ZZ*RECEIVERID     *230824*1200*U*00401*000000001*0*T*:~\n")
    f.write("GS*HC*SENDERID*RECEIVERID*20230824*1200*1*X*004010X098A1~\n")
    f.write("ST*837*0001~\n")

# 2. Mock Debezium CDC
with open("/tmp/mock_data/sample_cdc.json", "w") as f:
    cdc_record = {
        "payload": {
            "op": "c",
            "after": {"patient_id": "123", "name": "Jane Doe"}
        }
    }
    f.write(json.dumps(cdc_record) + "\n")

print("Mock files created.")

In [ ]:
import uuid
from pyspark.sql.functions import current_timestamp, lit, udf
from pyspark.sql.types import StringType

uuid_udf = udf(lambda: str(uuid.uuid4()), StringType())

# Process X12
df_x12 = spark.read.text("/tmp/mock_data/sample.edi")
df_bronze_x12 = df_x12.selectExpr("value as raw_payload_json") \
    .withColumn("payload_id", uuid_udf()) \
    .withColumn("source_system_id", lit("EDI_X12_MOCK")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "ingestion_timestamp", "raw_payload_json")

df_bronze_x12.write.format("iceberg").mode("append").saveAsTable("local.bronze_db.raw_payloads")

# Process CDC
df_cdc = spark.read.text("/tmp/mock_data/sample_cdc.json")
df_bronze_cdc = df_cdc.selectExpr("value as raw_payload_json") \
    .withColumn("payload_id", uuid_udf()) \
    .withColumn("source_system_id", lit("DEBEZIUM_CDC_MOCK")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .select("payload_id", "source_system_id", "ingestion_timestamp", "raw_payload_json")

df_bronze_cdc.write.format("iceberg").mode("append").saveAsTable("local.bronze_db.raw_payloads")

print("Batch processing simulated and written to Bronze.")

In [ ]:
# Verify Bronze Table
spark.sql("SELECT * FROM local.bronze_db.raw_payloads WHERE source_system_id IN ('EDI_X12_MOCK', 'DEBEZIUM_CDC_MOCK')").show(truncate=False)